# WHERE, LIMIT, ORDER BY, GROUP BY a logické operátory

V předchozích noteboocích jsme se naučili vkládat (`INSERT`) a číst (`SELECT`) data z tabulek. Příkaz `SELECT` však zatím vracel **všechny záznamy**. Nyní se naučíme výsledky **filtrovat**, **řadit**, **omezovat** a **seskupovat**.

---

## Přehled příkazů v tomto notebooku

| Klíčové slovo | Popis | Příklad |
|---|---|---|
| `WHERE` | Filtruje řádky podle podmínky | `WHERE max_rychlost > 200` |
| `AND` | Logický součin — musí platit **obě** podmínky | `WHERE a = 1 AND b = 2` |
| `OR` | Logický součet — musí platit **alespoň jedna** podmínka | `WHERE a = 1 OR b = 2` |
| `NOT` | Negace — vrátí řádky, které podmínku **nesplňují** | `WHERE NOT a = 1` |
| `ORDER BY` | Seřadí výsledky (vzestupně/sestupně) | `ORDER BY cena DESC` |
| `LIMIT` | Omezí počet vrácených řádků | `LIMIT 5` |
| `GROUP BY` | Seskupí řádky se stejnou hodnotou ve sloupci | `GROUP BY kategorie` |
| `HAVING` | Filtruje **skupiny** (jako `WHERE`, ale pro `GROUP BY`) | `HAVING COUNT(*) > 2` |

### Pořadí klíčových slov v dotazu

SQL příkazy musí být zapsány v **přesném pořadí**:

```sql
SELECT sloupce
FROM tabulka
WHERE podminka           -- filtruje řádky
GROUP BY sloupec         -- seskupí řádky
HAVING podminka_skupiny  -- filtruje skupiny
ORDER BY sloupec         -- seřadí výsledky
LIMIT pocet;             -- omezí počet řádků
```

> **⚠️ Pozor:** Pořadí je důležité — pokud napíšete `LIMIT` před `WHERE`, dostanete chybu.

## Instalace balíčku

In [ ]:
# Instalace knihovny (stačí spustit jednou)
! pip install mysql-connector-python

# Připojení k databázi

Než začneme s databází pracovat, musíme se k ní připojit. Přihlašovací údaje importujeme z modulu `pripojeni.py` (nikdy je nepíšeme přímo do kódu).

In [ ]:
import mysql.connector
from pripojeni import *  # importuje konstanty HOST, USER, PASSWORD, DATABASE

# Připojení k databázi
mydb = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DATABASE
)

# Kurzor — objekt, přes který posíláme SQL příkazy a čteme výsledky
mycursor = mydb.cursor()

## Příprava ukázkových dat

Pro následující příklady si vytvoříme a naplníme tabulku `automobily`:

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")

mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")

mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 180, 5, 'D', '2002-09-09'),
        ('4D44444', 2, 220, 2, 'B', '2020-05-01'),
        ('5E55555', 5, 180, 5, 'D', '2002-09-09'),
        ('6F66666', 8, 120, 10, 'C', '2015-06-15'),
        ('7G77777', 3, 250, 2, 'A', '2022-03-20'),
        ('8H88888', 4, 190, 3, 'B', '2018-11-30')
""")

mydb.commit()
print("✅ Tabulka 'automobily' vytvořena a naplněna (8 záznamů).")

---

# WHERE — filtrování řádků

Klíčové slovo `WHERE` slouží k **filtrování řádků** podle zadané podmínky. Zobrazí se pouze řádky, které podmínku splňují.

## Syntaxe

```sql
SELECT sloupce FROM tabulka WHERE podminka;
```

## Porovnávací operátory

| Operátor | Popis | Příklad |
|---|---|---|
| `=` | Rovná se | `WHERE id = 3` |
| `!=` nebo `<>` | Nerovná se | `WHERE id != 3` |
| `>` | Větší než | `WHERE max_rychlost > 200` |
| `<` | Menší než | `WHERE pocet_sedadel < 4` |
| `>=` | Větší nebo rovno | `WHERE nosnost >= 5` |
| `<=` | Menší nebo rovno | `WHERE nosnost <= 3` |

> **Pozn.:** Textové hodnoty v podmínce musí být v uvozovkách: `WHERE spz = '1A11111'`

### Příklad — filtrování podle konkrétní hodnoty

In [ ]:
# Vypíšeme auto s id = 3
mycursor.execute("""SELECT * FROM automobily WHERE id = 3""")

for id, spz, sedadel, rychlost, nosnost, kvalifikace, vyrobeno in mycursor.fetchall():
    print(f"Auto #{id}: SPZ={spz}, sedadel={sedadel}, rychlost={rychlost}, nosnost={nosnost}, kvalifikace={kvalifikace}, vyrobeno={vyrobeno}")

### Příklad — filtrování s porovnáním

In [ ]:
# Auta s více než 3 sedadly
mycursor.execute("""SELECT id, spz, pocet_sedadel FROM automobily WHERE pocet_sedadel > 3""")

print("Auta s více než 3 sedadly:")
for id, spz, sedadel in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, sedadel={sedadel}")

---

# Logické operátory — AND, OR, NOT

Podmínky v `WHERE` lze kombinovat pomocí logických operátorů:

| Operátor | Popis | Příklad |
|---|---|---|
| `AND` | Musí platit **obě** podmínky | `WHERE a = 1 AND b = 2` |
| `OR` | Musí platit **alespoň jedna** | `WHERE a = 1 OR b = 2` |
| `NOT` | **Neguje** podmínku | `WHERE NOT a = 1` |

> **Tip:** Operátory lze kombinovat a závorkami určovat prioritu: `WHERE (a = 1 OR b = 2) AND c = 3`

### Příklad — AND

In [ ]:
# Auta s kvalifikací 'B' A rychlostí nad 200
mycursor.execute("""
    SELECT id, spz, max_rychlost, nutna_kvalifikace
    FROM automobily
    WHERE nutna_kvalifikace = 'B' AND max_rychlost > 200
""")

print("Auta s kvalifikací B a rychlostí > 200:")
for id, spz, rychlost, kvalifikace in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, rychlost={rychlost}, kvalifikace={kvalifikace}")

### Příklad — OR

In [ ]:
# Auta s 2 sedadly NEBO kvalifikací 'D'
mycursor.execute("""
    SELECT id, spz, pocet_sedadel, nutna_kvalifikace
    FROM automobily
    WHERE pocet_sedadel = 2 OR nutna_kvalifikace = 'D'
""")

print("Auta s 2 sedadly nebo kvalifikací D:")
for id, spz, sedadel, kvalifikace in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, sedadel={sedadel}, kvalifikace={kvalifikace}")

### Příklad — NOT

In [ ]:
# Auta, která NEMAJÍ kvalifikaci 'B'
mycursor.execute("""
    SELECT id, spz, nutna_kvalifikace
    FROM automobily
    WHERE NOT nutna_kvalifikace = 'B'
""")

print("Auta bez kvalifikace B:")
for id, spz, kvalifikace in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, kvalifikace={kvalifikace}")

---

# ORDER BY — řazení výsledků

Příkaz `ORDER BY` slouží k **seřazení** výsledků podle jednoho nebo více sloupců.

## Syntaxe

```sql
SELECT sloupce FROM tabulka ORDER BY sloupec [ASC|DESC];
```

| Směr | Popis | Výchozí? |
|---|---|---|
| `ASC` | Vzestupně (od nejmenšího) | ✅ Ano |
| `DESC` | Sestupně (od největšího) | Ne |

Řadit lze i podle **více sloupců** — nejdříve se řadí podle prvního, při shodě podle druhého atd.

```sql
SELECT * FROM automobily ORDER BY nutna_kvalifikace ASC, max_rychlost DESC;
```

### Příklad

In [ ]:
# Seřadíme auta podle rychlosti sestupně
mycursor.execute("""
    SELECT id, spz, max_rychlost
    FROM automobily
    ORDER BY max_rychlost DESC
""")

print("Auta seřazená od nejrychlejšího:")
for id, spz, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, rychlost={rychlost} km/h")

In [ ]:
# Kombinace WHERE + ORDER BY
mycursor.execute("""
    SELECT id, spz, max_rychlost, nutna_kvalifikace
    FROM automobily
    WHERE nutna_kvalifikace = 'B'
    ORDER BY max_rychlost ASC
""")

print("Auta s kvalifikací B, od nejpomalejšího:")
for id, spz, rychlost, kvalifikace in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, rychlost={rychlost} km/h")

---

# LIMIT — omezení počtu výsledků

Příkaz `LIMIT` omezí počet vrácených řádků. Je užitečný pro **stránkování** nebo když chceme vidět jen prvních pár záznamů.

## Syntaxe

```sql
SELECT sloupce FROM tabulka LIMIT pocet;

/* S offsetem (přeskočení prvních X řádků) */
SELECT sloupce FROM tabulka LIMIT offset, pocet;
```

| Zápis | Význam |
|---|---|
| `LIMIT 3` | Vrátí **prvních 3** řádků |
| `LIMIT 2, 3` | Přeskočí 2 řádky a vrátí **následující 3** |

> **Tip:** `LIMIT` se často kombinuje s `ORDER BY` — např. „3 nejrychlejší auta


In [ ]:
mycursor.execute("""
    SELECT id, spz, max_rychlost
    FROM automobily
    ORDER BY max_rychlost DESC
    LIMIT 3
""")


print("Top 3 nejrychlejší auta:")
for id, spz, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, rychlost={rychlost} km/h")

In [ ]:
# Stránkování — přeskočíme 2 záznamy, zobrazíme 3
mycursor.execute("""
    SELECT id, spz, max_rychlost
    FROM automobily
    ORDER BY id
    LIMIT 2, 3
""")

print("Záznamy 3–5 (stránkování):")
for id, spz, rychlost in mycursor.fetchall():
    print(f"  Auto #{id}: SPZ={spz}, rychlost={rychlost} km/h")

---

# GROUP BY — seskupení řádků

Příkaz `GROUP BY` seskupí řádky, které mají **stejnou hodnotu** v zadaném sloupci. Typicky se používá společně s **agregačními funkcemi** (`COUNT`, `SUM`, `AVG`, …).

## Syntaxe

```sql
SELECT sloupec, AGREGACE(sloupec2) FROM tabulka GROUP BY sloupec;
```

### Co GROUP BY dělá?

1. Vezme všechny řádky z tabulky.
2. Seskupí je podle zadaného sloupce (řádky se stejnou hodnotou = jedna skupina).
3. Pro každou skupinu spočítá agregační funkci.

> **Pravidlo:** V `SELECT` mohou být pouze sloupce uvedené v `GROUP BY` a agregační funkce. Jiné sloupce by neměly smysl — ze skupiny by nebylo jasné, kterou hodnotu zobrazit.

### Příklad — unikátní hodnoty

In [ ]:
# Jaké různé kvalifikace se v tabulce vyskytují?
mycursor.execute("""
    SELECT nutna_kvalifikace
    FROM automobily
    GROUP BY nutna_kvalifikace
""")

print("Unikátní kvalifikace:")
for (kvalifikace,) in mycursor.fetchall():
    print(f"  {kvalifikace}")

### Příklad — GROUP BY s agregační funkcí

In [ ]:
# Kolik aut je pro každou kvalifikaci?
mycursor.execute("""
    SELECT nutna_kvalifikace, COUNT(*) AS 'pocet_aut'
    FROM automobily
    GROUP BY nutna_kvalifikace
""")

print("Počet aut podle kvalifikace:")
for kvalifikace, pocet in mycursor.fetchall():
    print(f"  Kvalifikace {kvalifikace}: {pocet} aut")

In [ ]:
# Průměrná rychlost a počet aut podle počtu sedadel
mycursor.execute("""
    SELECT pocet_sedadel, COUNT(*) AS 'pocet', AVG(max_rychlost) AS 'prumerna_rychlost'
    FROM automobily
    GROUP BY pocet_sedadel
    ORDER BY pocet_sedadel
""")

print("Statistiky podle počtu sedadel:")
for sedadel, pocet, prumer in mycursor.fetchall():
    print(f"  {sedadel} sedadel: {pocet} aut, průměrná rychlost {prumer} km/h")

## HAVING — filtrování skupin

`HAVING` funguje jako `WHERE`, ale filtruje **skupiny** vytvořené pomocí `GROUP BY` (ne jednotlivé řádky).

```sql
SELECT sloupec, COUNT(*) FROM tabulka
GROUP BY sloupec
HAVING COUNT(*) > 2;
```

> **Rozdíl WHERE vs. HAVING:**
> - `WHERE` filtruje řádky **před** seskupením.
> - `HAVING` filtruje skupiny **po** seskupení.

### Příklad

In [ ]:
# Kvalifikace, pro které máme více než 1 auto
mycursor.execute("""
    SELECT nutna_kvalifikace, COUNT(*) AS 'pocet'
    FROM automobily
    GROUP BY nutna_kvalifikace
    HAVING COUNT(*) > 1
""")

print("Kvalifikace s více než 1 autem:")
for kvalifikace, pocet in mycursor.fetchall():
    print(f"  Kvalifikace {kvalifikace}: {pocet} aut")

---

# Kombinace všech příkazů

Všechny příkazy lze kombinovat v jednom dotazu (ve správném pořadí):

### Příklad

In [ ]:
# Kvalifikace s průměrnou rychlostí > 150, ale jen pro auta vyrobená po roce 2000,
# seřazené sestupně podle průměrné rychlosti, zobrazíme max. 3 výsledky
mycursor.execute("""
    SELECT
        nutna_kvalifikace,
        COUNT(*) AS 'pocet',
        AVG(max_rychlost) AS 'prumerna_rychlost'
    FROM automobily
    WHERE datum_vyroby > '2000-01-01'
    GROUP BY nutna_kvalifikace
    HAVING AVG(max_rychlost) > 150
    ORDER BY prumerna_rychlost DESC
    LIMIT 3
""")

print("Výsledek kombinovaného dotazu:")
for kvalifikace, pocet, prumer in mycursor.fetchall():
    print(f"  Kvalifikace {kvalifikace}: {pocet} aut, průměr {prumer} km/h")

## Odpojení a úklid

Na konci ukázkové části smažeme tabulku a odpojíme se:

In [ ]:
mycursor.execute("DROP TABLE IF EXISTS automobily")
mydb.commit()

mycursor.close()
mydb.close()

print("✅ Odpojení proběhlo úspěšně.")

---

# Cvičení

Zde bude následovat série úkolů, díky kterým si ověříte, zda jste látku pochopili.

> V každém cvičení se nejprve připojte k databázi pomocí konstant z modulu `pripojeni`. Na konci se vždy odpojte.

## Cvičení 1 — LIMIT

Z tabulky `automobily` (vytvořené v kódu níže) vypište **pouze první 3 záznamy**.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #1: SPZ=1A11111, sedadel=4, rychlost=190 km/h
Auto #2: SPZ=2B22222, sedadel=2, rychlost=220 km/h
Auto #3: SPZ=3C33333, sedadel=5, rychlost=180 km/h
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 180, 5, 'D', '2002-09-09'),
        ('4D44444', 2, 220, 2, 'B', '2020-05-01'),
        ('5E55555', 5, 180, 5, 'D', '2002-09-09'),
        ('6F66666', 8, 120, 10, 'C', '2015-06-15')
""")
mydb.commit()

# TODO: Vypište první 3 záznamy z tabulky automobily →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 2 — GROUP BY

Z tabulky `automobily` vypište **unikátní hodnoty** sloupce `pocet_sedadel` pomocí `GROUP BY`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Počet sedadel: 2
Počet sedadel: 4
Počet sedadel: 5
Počet sedadel: 8
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 180, 5, 'D', '2002-09-09'),
        ('4D44444', 2, 220, 2, 'B', '2020-05-01'),
        ('5E55555', 5, 180, 5, 'D', '2002-09-09'),
        ('6F66666', 8, 120, 10, 'C', '2015-06-15')
""")
mydb.commit()

# TODO: Vypište unikátní hodnoty pocet_sedadel pomocí GROUP BY →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 3 — WHERE

Z tabulky `automobily` vypište záznamy, které mají `datum_vyroby = '2002-09-09'`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Auto #3: SPZ=3C33333, sedadel=5, rychlost=180, kvalifikace=D, vyrobeno=2002-09-09
Auto #5: SPZ=5E55555, sedadel=5, rychlost=180, kvalifikace=D, vyrobeno=2002-09-09
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 180, 5, 'D', '2002-09-09'),
        ('4D44444', 2, 220, 2, 'B', '2020-05-01'),
        ('5E55555', 5, 180, 5, 'D', '2002-09-09'),
        ('6F66666', 8, 120, 10, 'C', '2015-06-15')
""")
mydb.commit()

# TODO: Vypište záznamy s datum_vyroby = '2002-09-09' →


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 4 — Logické operátory

Z tabulky `automobily` postupně vypište:

**a)** Auta, která mají `pocet_sedadel = 2` **NEBO** `nutna_kvalifikace = 'D'` — vypište pouze `spz` a `pocet_sedadel`.

**b)** Auta s `max_rychlost > 200` **A** `nutna_kvalifikace = 'B'` — vypište pouze `spz` a `max_rychlost`.

**c)** Auta, která **NEMAJÍ** `nutna_kvalifikace = 'D'` — vypište pouze `id` a `nutna_kvalifikace`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- a) OR ---
SPZ=2B22222, sedadel=2
SPZ=3C33333, sedadel=5
SPZ=4D44444, sedadel=2
SPZ=5E55555, sedadel=5

--- b) AND ---
SPZ=2B22222, rychlost=220
SPZ=4D44444, rychlost=220

--- c) NOT ---
Auto #1, kvalifikace=B
Auto #2, kvalifikace=B
Auto #4, kvalifikace=B
Auto #6, kvalifikace=C
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# --- tuto část neměnit! (příprava dat) ---
mycursor.execute("DROP TABLE IF EXISTS automobily")
mycursor.execute("""
    CREATE TABLE automobily (
        id INT PRIMARY KEY AUTO_INCREMENT,
        spz CHAR(7) NOT NULL,
        pocet_sedadel INT NOT NULL,
        max_rychlost INT,
        nosnost INT,
        nutna_kvalifikace VARCHAR(10) NOT NULL DEFAULT 'B',
        datum_vyroby DATE
    )
""")
mycursor.execute("""
    INSERT INTO automobily (spz, pocet_sedadel, max_rychlost, nosnost, nutna_kvalifikace, datum_vyroby) VALUES
        ('1A11111', 4, 190, 3, 'B', '2000-09-09'),
        ('2B22222', 2, 220, 2, 'B', '2020-01-01'),
        ('3C33333', 5, 180, 5, 'D', '2002-09-09'),
        ('4D44444', 2, 220, 2, 'B', '2020-05-01'),
        ('5E55555', 5, 180, 5, 'D', '2002-09-09'),
        ('6F66666', 8, 120, 10, 'C', '2015-06-15')
""")
mydb.commit()

# TODO: a) OR →
print("--- a) OR ---")


# TODO: b) AND →
print("\n--- b) AND ---")


# TODO: c) NOT →
print("\n--- c) NOT ---")


# --- tuto část neměnit! (úklid) ---
mycursor.execute("DROP TABLE automobily")
mydb.commit()
mycursor.close()
mydb.close()

## Cvičení 5 — Kompletní úloha

Celý úkol je na vás od začátku do konce:

1. Připojte se k databázi.
2. Vytvořte tabulku `zamestnanci` s atributy:
   - `id` — INT, PRIMARY KEY, AUTO_INCREMENT
   - `jmeno` — VARCHAR(50), NOT NULL
   - `prijmeni` — VARCHAR(50), NOT NULL
   - `oddeleni` — VARCHAR(30), NOT NULL
   - `plat` — INT, NOT NULL
   - `datum_nastupu` — DATE
3. Vložte **alespoň 8 zaměstnanců** ve **3 různých odděleních** s různými platy.
4. Vypište:
   - **3 nejlépe placené** zaměstnance (`ORDER BY` + `LIMIT`)
   - Zaměstnance s platem **nad 40 000** (`WHERE`)
   - Zaměstnance z oddělení `'IT'` **nebo** s platem **nad 50 000** (`WHERE` + `OR`)
   - **Počet zaměstnanců a průměrný plat** v každém oddělení (`GROUP BY` + agregace)
   - Oddělení, která mají **více než 2 zaměstnance** (`GROUP BY` + `HAVING`)
5. Smažte tabulku a odpojte se.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Top 3 platy ---
Jan Novak (IT) — 65000 Kč
...

--- Plat nad 40 000 ---
...

--- IT nebo plat > 50 000 ---
...

--- Statistiky podle oddělení ---
IT: 3 zaměstnanců, průměrný plat: 52333 Kč
...

--- Oddělení s > 2 zaměstnanci ---
...
```

</details>

In [ ]:
import mysql.connector
from pripojeni import *

mydb = mysql.connector.connect(
    host=HOST, user=USER, password=PASSWORD, database=DATABASE
)
mycursor = mydb.cursor()

# TODO: Celý úkol je na vás →


# Nezapomeňte na konci smazat tabulku a odpojit se!